## Train a Variational Autoencoder using MNIST

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal
import numpy as np
import torch.utils.data as data
from torchvision.utils import make_grid
from torchvision import datasets, transforms
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm.notebook import tqdm, trange

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
print("Device:", device)

### Prepare the Data

In [ ]:
transform = transforms.Compose([transforms.ToTensor()
                               ])

mnist_trainset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
mnist_testset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

In [ ]:
train_data = mnist_trainset.data.unsqueeze(1).float() / 255.0
train_data = train_data.to(device)
test_data = mnist_testset.data.unsqueeze(1).float() / 255.0
test_data = test_data.to(device)
test_targets = mnist_testset.targets

In [ ]:
train_dataset = TensorDataset(train_data)
test_dataset = TensorDataset(test_data)

In [ ]:
train_batch_size = 128
test_batch_size = 256
train_loader = DataLoader(train_dataset, batch_size=train_batch_size, 
                               shuffle=True,  drop_last=True)
test_loader  = DataLoader(test_dataset,  batch_size=test_batch_size, 
                               shuffle=False, drop_last=False)

In [ ]:
def show_tensor_images(image_tensor, 
                       num_images=64, 
                       size=(1, 32, 32), 
                       nrow=8, 
                       filename=""):
    image_unflat = image_tensor.detach().cpu().view(-1, *size)
    image_grid = make_grid(image_unflat[:num_images], nrow=nrow)
    img = image_grid.permute(1, 2, 0).squeeze()
    plt.axis('off')
    plt.imshow(img)
    if len(filename) > 0:
        plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()

### Build the Autoencoder

In [ ]:
class Sampling(nn.Module):
    def forward(self, inputs):
        mu, log_var = inputs
        std = torch.exp(0.5 * log_var)
        epsilon = torch.randn_like(std)
        return mu + std * epsilon

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super(Encoder, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=input_dim[0], out_channels=32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.conv3 = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(32)
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(800, 16)
        self.mu = nn.Linear(16, latent_dim)
        self.log_var = nn.Linear(16, latent_dim)
        self.sampling = Sampling()

    def forward(self, x):
        x = F.silu(self.bn1(self.conv1(x)))
        x = F.max_pool2d(x, kernel_size=2, padding=1)
        x = F.silu(self.bn2(self.conv2(x)))
        x = F.max_pool2d(x, kernel_size=2, padding=1)
        x = F.silu(self.bn3(self.conv3(x)))
        x = F.max_pool2d(x, kernel_size=2, padding=1)
        x = self.flatten(x)
        x = F.relu(self.fc(x))
        mu = self.mu(x)
        log_var = self.log_var(x)
        z = self.sampling((mu, log_var))
        return mu, log_var, z

In [ ]:
class Decoder(nn.Module):
    def __init__(self, latent_dim):
        super(Decoder, self).__init__()
        self.fc = nn.Linear(latent_dim, 7 * 7 * 32)
        self.deconv1 = nn.ConvTranspose2d(32, 32, kernel_size=3, stride=2, padding=1, output_padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.deconv2 = nn.ConvTranspose2d(32, 32, kernel_size=3, stride=2, padding=1, output_padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.deconv3 = nn.ConvTranspose2d(32, 1, kernel_size=3, padding=1)
    
    def forward(self, x):
        x = F.relu(self.fc(x))
        x = x.view(-1, 32, 7, 7)
        x = F.silu(self.bn1(self.deconv1(x)))
        x = F.silu(self.bn2(self.deconv2(x)))
        x = torch.sigmoid(self.deconv3(x))
        return x

In [ ]:
class VAE(nn.Module):
    def __init__(self, encoder, decoder, beta=1.0):
        super(VAE, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.beta = beta

    def forward(self, x):
        mu, log_var, z = self.encoder(x)
        reconstruction = self.decoder(z)
        return reconstruction, mu, log_var

In [ ]:
input_dim = (1, 28, 28)
latent_dim = 3

In [ ]:
encoder = Encoder(input_dim=input_dim, latent_dim=latent_dim).to(device)
decoder = Decoder(latent_dim=latent_dim).to(device)
vae = VAE(encoder, decoder).to(device)

In [ ]:
learning_rate = 0.001
optimizer = torch.optim.Adam(vae.parameters(), lr=learning_rate)

In [ ]:
num_epochs = 50
beta = 1

In [ ]:
def vae_loss(recon_x, x, mean, log_var, beta):
    recon_loss = F.binary_cross_entropy(recon_x.view(-1, 784), x.view(-1, 784), reduction='sum')
    kld_loss = -0.5 * torch.sum(1 + log_var - mean ** 2 - log_var.exp())
    return recon_loss * beta + kld_loss, recon_loss, kld_loss

In [ ]:
loss_lst = list()
reloss_list = list()
kl_loss_list = list()
tqdm_epoch = trange(num_epochs)

for epoch in tqdm_epoch:
    sum_loss = 0.0
    num_items = 0
    re_loss = 0.0
    kl_loss = 0.0
    for [data] in train_loader:
        data = data.to(device)
        cur_batch_size = len(data)
        optimizer.zero_grad()
        recon_batch, mean, log_var = vae(data)
        loss, rc_loss, kl_loss = vae_loss(recon_batch, data, mean, log_var, beta)
        loss.backward()
        optimizer.step()
        
        sum_loss += loss.item() * cur_batch_size
        re_loss += rc_loss.item() * cur_batch_size
        kl_loss += kl_loss.item() * cur_batch_size
        num_items += cur_batch_size
    
    ave_loss = sum_loss / num_items
    ave_re_loss = re_loss / num_items
    ave_kl_loss = kl_loss / num_items
    loss_lst.append(ave_loss)
    tqdm_epoch.set_description('Ave Loss: {:5f}, Ave Rec Loss: {:5f},'
                               'Ave KL Loss: {:5f}'.format(ave_loss, ave_re_loss, ave_kl_loss))


In [ ]:
torch.save(vae.state_dict(), 'vae_genckpt.pth')

### Display Reconstructions

In [ ]:
test_z = encoder(test_data)
reconstruction = decoder(test_z[2])

In [ ]:
plt.figure(figsize=(10, 20))
show_tensor_images(reconstruction, 
                       num_images=20, 
                       size=(1, 28, 28), 
                       nrow=20, 
                       filename="vaereconstruct.png")

In [ ]:
plt.figure(figsize=(10, 20))
show_tensor_images(test_data,
                   num_images=20, 
                   size=(1, 28, 28), 
                   nrow=20,  
                   filename='vaetest.png')

### Display the Latent Space

In [ ]:
latent_pts = vae.encoder(test_data[0:1000])

In [ ]:
latent_pts[2]

In [ ]:
nlatent_pts = latent_pts[2].detach().cpu().numpy()

In [ ]:
test_targets

In [ ]:
fig = plt.figure(figsize=(20, 20))
ax = fig.add_subplot(projection='3d')

for i, x in enumerate(nlatent_pts):
    ax.scatter(x[0], x[1], x[2], color='k')
    ax.text(x[0], x[1], x[2], '%s' % (test_targets.numpy()[i]), size=20, zorder=1, color='k')

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
plt.savefig('vae3dscatter.png', dpi=300, bbox_inches='tight')

### Generate samples

In [ ]:
sample_z = torch.randn(64, 3).to(device)

In [ ]:
decoded_samples = vae.decoder(sample_z)

In [ ]:
plt.figure(figsize=(10, 20))
show_tensor_images(decoded_samples, 
                       num_images=40, 
                       size=(1, 28, 28), 
                       nrow=20, 
                       filename="VAEGenerationGrid.png")